In [1]:
import os
import json
import shutil
import sys
import time

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.regularizers import (
    FastFixPhiRegularizer, DecorrelateWithOtherPhiRegularizer, DecorrelateWithOtherPhiRegularizer2
)
from topnum.scores.intratext_coherence_score import (
    IntratextCoherenceScore,
    ComputationMethod,
    WordTopicRelatednessType,
)
from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [5]:
DATA_FOLDER_PATH = '/data_mil/shared/CompressaAI/iterative/data/noow'

In [6]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/20_Newsgroups_NOOW.csv',
)

dataset.get_possible_modalities()

{'@lemmatized'}

In [7]:
MAIN_MODALITY = '@lemmatized'

In [8]:
dataset._data.head()

,raw_text,id,vw_text
id,,,
rec_autos_102994,I was wondering if anyone out there could enli...,rec_autos_102994,rec_autos_102994 |@lemmatized wonder anyone co...
comp_sys_mac_hardware_51861,A fair number of brave souls who upgraded thei...,comp_sys_mac_hardware_51861,comp_sys_mac_hardware_51861 |@lemmatized fair ...
comp_sys_mac_hardware_51879,"well folks, my mac plus finally gave up the gh...",comp_sys_mac_hardware_51879,comp_sys_mac_hardware_51879 |@lemmatized well ...
comp_graphics_38242,\nDo you have Weitek's address/phone number? ...,comp_graphics_38242,comp_graphics_38242 |@lemmatized weitek addres...
sci_space_60880,"From article <C5owCB.n3p@world.std.com>, by to...",sci_space_60880,sci_space_60880 |@lemmatized article tom baker...


In [9]:
dataset.get_dictionary()

artm.Dictionary(name=3e6c5a2d-89aa-4d86-abc0-004d54a2a182, num_entries=52744)

In [10]:
dictionary = dataset.get_dictionary()

print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

print(dictionary)

artm.Dictionary(name=3e6c5a2d-89aa-4d86-abc0-004d54a2a182, num_entries=52744)
artm.Dictionary(name=3e6c5a2d-89aa-4d86-abc0-004d54a2a182, num_entries=52744)


In [11]:
dataset._cached_dict = dictionary

In [12]:
dictionary = dataset.get_dictionary()

In [13]:
dictionary

artm.Dictionary(name=3e6c5a2d-89aa-4d86-abc0-004d54a2a182, num_entries=52744)

In [14]:
def is_good(coherence):
    return 1.7456368947464058 <= coherence

def is_bad(coherence):
    return coherence <= 1.3760632784442917

In [15]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [16]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 3.36 s, sys: 125 ms, total: 3.49 s
Wall time: 3.45 s


In [17]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [18]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [19]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [20]:
NUM_TOPICS = 50  # vary
MAX_NUM_TRAINS = 20
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

In [21]:
NUM_GOOD_TOPICS_THRESHOLD = NUM_TOPICS - 5

In [22]:
def fit_and_compute_scores(model, dataset, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}



    # intra1 = IntratextCoherenceScore(
    #     name='toplen_pwt',
    #     data=dataset,
    #     computation_method=ComputationMethod.SEGMENT_LENGTH,
    #     word_topic_relatedness=WordTopicRelatednessType.PWT,
    #     should_compute=False,  # only on last iter
    # )
    intra2 = IntratextCoherenceScore(
        name='toplen_ptw',
        data=dataset,
        computation_method=ComputationMethod.SEGMENT_LENGTH,
        word_topic_relatedness=WordTopicRelatednessType.PTW,
        should_compute=False,
    )
    # intra3 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     should_compute=False,
    # )
    # intra3_w4 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     window=4,
    #     should_compute=False,
    # )

    intra_topic_coherences = dict()

    for intra in [intra2]:  # [intra2, intra3_w4]:  #[intra1, intra2, intra3]:
        # print(f'\nComputing "{intra._name}"...')

        current_intra_topic_coherences = intra.compute(model)

        assert all(v is not None for v in current_intra_topic_coherences.values())

        _values = current_intra_topic_coherences.values()

        current_intra_topic_coherences = {
            i: current_intra_topic_coherences[t]  # if v is not None else 0.0
            for i, t in enumerate(target_topic_names)
        }

        assert all(abs(x - y) <= 1e-6 for x, y in zip(_values, current_intra_topic_coherences.values())), (_values, current_intra_topic_coherences.values())  # "sorted" Python dicts
        
        intra_topic_coherences[f'topic_coherences_{intra._name}'] = current_intra_topic_coherences

        value = float(np.median(list(current_intra_topic_coherences.values())))
        score_values[intra._name] = value

        # print(f'Result by topic: {current_intra_topic_coherences}.')


    
    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
        **intra_topic_coherences,
    }

In [23]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [24]:
NUM_TRAINS = 3
TOPIC_INDICES = list(range(NUM_TOPICS))

In [25]:
TOPIC_INDICES

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49]

In [26]:
! ls results

ls: cannot access 'results': No such file or directory


In [27]:
! ls results/20newsgroups/

ls: cannot access 'results/20newsgroups/': No such file or directory


In [28]:
BEST_TAUS = [100000, 100000000]

In [53]:
! ls results50

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [33]:
SAVE_FOLDER = 'results50_intra/20newsgroups'

os.makedirs(SAVE_FOLDER, exist_ok=True)

In [39]:
SAVE_FOLDER

'results50_intra/20newsgroups'

In [35]:
! ls results50_intra

20newsgroups  mkb10  postnauka


In [55]:
BEST_TAUS

[100000, 100000000]

In [31]:
! ls results50_iter

20newsgroups


In [44]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelateWithOtherPhiRegularizer

DECORRELATION_TAUS = [BEST_TAUS[0]]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    
    res_file_path = SAVE_FOLDER + f'/iterative_{int(key)}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue
    
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        
        seed_save_folder = os.path.join(SAVE_FOLDER, f'iterative_{key}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'iterative_{key}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:

            print('test_1')

            if good_phi is None:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
            else:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_phi=good_phi,
                    topic_names=good_topic_names,
                )

            print('test_2')
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            print('test_3')
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )

            print('test_4')
            
            good_phi = prev_model._model.get_phi()[good_topic_names]  # TODO: no prev_model here after loading (AttributeError: 'NoneType' object has no attribute '_model')
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )

            print('test_5')
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            print('test_6')
            
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }

            print('test_7')
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break

            print('test_8')
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]



            assert not any(t in new_bad_topic_names for t in good_topic_names)
            assert np.allclose(
                prev_model.get_phi()[good_topic_names].to_numpy(),
                phi[good_topic_names].to_numpy(),
                atol=1e-5
            )

            good_fair = len(new_good_topic_names)

            if not (set(good_topic_names) <= set(new_good_topic_names)):
                assert any(t in new_not_good_topic_names for t in good_topic_names)
                assert not any(t in new_bad_topic_names for t in good_topic_names)

                print(
                    f'DOWNFALL: some old good topics {good_topic_names}'
                    f' are not in new good topics {new_good_topic_names}.'
                    f' Manually marking them as good.'
                )

                # new_good_topic_names = list(
                #     set(new_good_topic_names).union(set(good_topic_names))
                # )
                new_expected_good_len = len(set(new_good_topic_names).union(set(good_topic_names)))

                new_good_topic_names = [
                    t for t in phi.columns
                    if t in new_good_topic_names or t in good_topic_names
                ]

                assert len(new_good_topic_names) == new_expected_good_len

                # new_not_good_topic_names = list(
                #     set(new_not_good_topic_names).difference(set(good_topic_names))
                # )
                new_expected_not_good_len = len(set(new_not_good_topic_names).difference(set(good_topic_names)))
                
                new_not_good_topic_names = [
                    t for t in phi.columns
                    if t in new_not_good_topic_names and t not in good_topic_names
                ]

                assert len(new_not_good_topic_names) == new_expected_not_good_len

                good_topic_indices = [phi.columns.get_loc(t) for t in new_good_topic_names]
                not_good_topic_indices = [phi.columns.get_loc(t) for t in new_not_good_topic_names]


            
            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': good_fair,
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            del prev_model
            
            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        with open(SAVE_FOLDER + f'/iterative_{int(k)}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

100000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.019165258246480757
sparse_theta_sp: -0.08944804715975409
decorrelation: 0.01
None
num_topics: {'good': 9, 'good_fair': 9, 'bad': 7, 'not_good': 41, 'total_bad': 7}
1
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f8cd5234be0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f8cfd05f910>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f8d242e3d90>}
test_8
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.023372266154244824
sparse_theta_sp: -0.10908298434116351
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
DOWNFALL: some old good topics ['topic_8', 'topic_19', 'topic_23', 'topic_28', 'topic_30', 'topic_33', 'topic_36', 'topic_39', 'topic_41'] are not in new good topics ['topic_8', 'topic_10', 'topic_19', 'topic_23', 'topic_26', 'topic_28', 'topic_33', 'topic_36', 'topic_39', 'topic_41', 'topic_46']. Manually marking them as good.
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 1

...

test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f8cfd3150d0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f8cfd05f1c0>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f8cfce904c0>}
test_8
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.03091170684916251
sparse_theta_sp: -0.144271043806055
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
DOWNFALL: some old good topics ['topic_8', 'topic_10', 'topic_13', 'topic_17', 'topic_19', 'topic_21', 'topic_22', 'topic_23', 'topic_26', 'topic_28', 'topic_30', 'topic_33', 'topic_34', 'topic_36', 'topic_39', 'topic_41', 'topic_42', 'topic_43', 'topic_46'] are not in new good topics ['topic_8', 'topic_10', 'topic_17', 'topic_19', 'topic_21', 'topic_23', 'topic_26', 'topic_28', 'topic_33', 'topic_34', 'top

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f8cfd7aefd0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f8aaa9c6490>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f8cb281e040>}
test_8
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.03091170684916251
sparse_theta_sp: -0.144271043806055
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
DOWNFALL: some old good topics ['topic_8', 'topic_10', 'topic_13', 'topic_17', 'topic_19', 'topic_21', 'topic_22', 'topic_23', 'topic_26', 'topic_28', 'topic_30', 'topic_33', 'topic_34', 'topic_36', 'topic_39', 'topic_41', 'topic_42', 'topic_43', 'topic_46'] are not in new good topics ['topic_8', 'topic_10', 'topic_17', 'topic_19', 'topic_21', 'topic_23', 'topic_26', 'topic_28', 'topic_33', 'topic_34', 'top

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f8cfd3150d0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f8cfd5a2df0>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f8cb281e610>}
test_8
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.03091170684916251
sparse_theta_sp: -0.144271043806055
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
DOWNFALL: some old good topics ['topic_8', 'topic_10', 'topic_13', 'topic_17', 'topic_19', 'topic_21', 'topic_22', 'topic_23', 'topic_26', 'topic_28', 'topic_30', 'topic_33', 'topic_34', 'topic_36', 'topic_39', 'topic_41', 'topic_42', 'topic_43', 'topic_46'] are not in new good topics ['topic_8', 'topic_10', 'topic_17', 'topic_19', 'topic_21', 'topic_23', 'topic_26', 'topic_28', 'topic_33', 'topic_34', 'top

In [46]:
1

1

In [47]:
results.keys()

dict_keys([100000])

In [48]:
SAVE_FOLDER

'results50_intra/20newsgroups'

In [49]:
! ls $SAVE_FOLDER

decorrelation_with_cohs.json  lda_with_cohs.json     tless_with_cohs.json
iterative_100000	      plsa_with_cohs.json
iterative_100000.json	      sparse_with_cohs.json


In [210]:
! ls $SAVE_FOLDER

decorrelation.json     iterative_10000.json  plsa.json	  tless.json
iterative_100000.json  lda.json		     sparse.json


In [50]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelateWithOtherPhiRegularizer2

DECORRELATION_TAUS = [BEST_TAUS[1]]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    
    res_file_path = SAVE_FOLDER + f'/iterative2_{int(key)}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue
    
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        
        seed_save_folder = os.path.join(SAVE_FOLDER, f'iterative2_{key}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'iterative2_{key}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:

            print('test_1')

            if good_phi is None:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
            else:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_phi=good_phi,
                    topic_names=good_topic_names,
                )

            print('test_2')
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            print('test_3')
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )

            print('test_4')
            
            good_phi = prev_model._model.get_phi()[good_topic_names]  # TODO: no prev_model here after loading (AttributeError: 'NoneType' object has no attribute '_model')
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )

            print('test_5')
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            print('test_6')
            
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }

            print('test_7')
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break

            print('test_8')
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]



            assert not any(t in new_bad_topic_names for t in good_topic_names)
            assert np.allclose(
                prev_model.get_phi()[good_topic_names].to_numpy(),
                phi[good_topic_names].to_numpy(),
                atol=1e-5
            )

            good_fair = len(new_good_topic_names)

            if not (set(good_topic_names) <= set(new_good_topic_names)):
                assert any(t in new_not_good_topic_names for t in good_topic_names)
                assert not any(t in new_bad_topic_names for t in good_topic_names)

                print(
                    f'DOWNFALL: some old good topics {good_topic_names}'
                    f' are not in new good topics {new_good_topic_names}.'
                    f' Manually marking them as good.'
                )

                # new_good_topic_names = list(
                #     set(new_good_topic_names).union(set(good_topic_names))
                # )
                new_expected_good_len = len(set(new_good_topic_names).union(set(good_topic_names)))

                new_good_topic_names = [
                    t for t in phi.columns
                    if t in new_good_topic_names or t in good_topic_names
                ]

                assert len(new_good_topic_names) == new_expected_good_len

                # new_not_good_topic_names = list(
                #     set(new_not_good_topic_names).difference(set(good_topic_names))
                # )
                new_expected_not_good_len = len(set(new_not_good_topic_names).difference(set(good_topic_names)))
                
                new_not_good_topic_names = [
                    t for t in phi.columns
                    if t in new_not_good_topic_names and t not in good_topic_names
                ]

                assert len(new_not_good_topic_names) == new_expected_not_good_len

                good_topic_indices = [phi.columns.get_loc(t) for t in new_good_topic_names]
                not_good_topic_indices = [phi.columns.get_loc(t) for t in new_not_good_topic_names]


            
            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': good_fair,
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            del prev_model
            
            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        with open(SAVE_FOLDER + f'/iterative2_{int(k)}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

100000000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.019165258246480757
sparse_theta_sp: -0.08944804715975409
decorrelation: 0.01
None
num_topics: {'good': 9, 'good_fair': 9, 'bad': 7, 'not_good': 41, 'total_bad': 7}
1
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f8cfcd90250>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f8cfd74beb0>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f8a89318220>}
test_8
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.023372266154244824
sparse_theta_sp: -0.10908298434116351
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 12, 'good_fair': 12, 'bad': 12, 'not_good': 38, 'total_bad': 19}
Removing: results50_intra/20newsgroups/iterative2_100000000/0
2
test_1
test_2
test_3
test_4
test_5


...

test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f8cb2819fd0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f8a86b5da30>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f8cb2819d60>}
test_8
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.02903827007042539
sparse_theta_sp: -0.13552734418144557
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some old good topics ['topic_6', 'topic_8', 'topic_10', 'topic_11', 'topic_13', 'topic_14', 'topic_19', 'topic_23', 'topic_26', 'topic_28', 'topic_30', 'topic_33', 'topic_36', 'topic_39', 'topic_41', 'topic_46', 'topic_47'] are not in new good topics ['topic_8', 'topic_10', 'topic_11', 'topic_13', 'topic_14', 'topic_19', 'topic_23', 'topic_26', 'topic_28', 'topic_33', 'topic_36', 'topic_

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f897cdae790>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f8a6cdb8fd0>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f8cb28199d0>}
test_8
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.02903827007042539
sparse_theta_sp: -0.13552734418144557
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some old good topics ['topic_6', 'topic_8', 'topic_10', 'topic_11', 'topic_13', 'topic_14', 'topic_19', 'topic_23', 'topic_26', 'topic_28', 'topic_30', 'topic_33', 'topic_36', 'topic_39', 'topic_41', 'topic_46', 'topic_47'] are not in new good topics ['topic_6', 'topic_8', 'topic_10', 'topic_11', 'topic_14', 'topic_19', 'topic_23', 'topic_26', 'topic_28', 'topic_33', 'topic_36', 'topic_3

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f89a7a9a520>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f89a3fc20a0>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f8989ac7610>}
test_8
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.02903827007042539
sparse_theta_sp: -0.13552734418144557
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some old good topics ['topic_6', 'topic_8', 'topic_10', 'topic_11', 'topic_13', 'topic_14', 'topic_19', 'topic_23', 'topic_26', 'topic_28', 'topic_30', 'topic_33', 'topic_36', 'topic_39', 'topic_41', 'topic_46', 'topic_47'] are not in new good topics ['topic_4', 'topic_6', 'topic_8', 'topic_10', 'topic_11', 'topic_14', 'topic_19', 'topic_23', 'topic_26', 'topic_28', 'topic_33', 'topic_36

In [51]:
1

1

In [63]:
results.keys()

dict_keys([100000000])

In [52]:
! ls $SAVE_FOLDER

decorrelation_with_cohs.json  iterative2_100000000	 plsa_with_cohs.json
iterative_100000	      iterative2_100000000.json  sparse_with_cohs.json
iterative_100000.json	      lda_with_cohs.json	 tless_with_cohs.json


In [53]:
good_phi.head()

,topic_4,topic_6,topic_8,topic_10,topic_11,topic_13,topic_14,topic_19,topic_23,topic_26,topic_28,topic_30,topic_33,topic_36,topic_39,topic_41,topic_46,topic_47
willow,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
bodin,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
aneurysm,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000044,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
millie,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000091,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
paroxysmal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000044,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [56]:
good_phi.shape

(52744, 18)

In [55]:
len(good_topic_names)

18

In [59]:
good_phi.iloc[:, 2].sort_values(ascending=False)[:20]  # TODO: background-like

think     0.035116
get       0.032095
would     0.028787
good      0.024501
like      0.023961
one       0.018709
well      0.017561
go        0.017444
make      0.016704
see       0.015935
thing     0.015619
say       0.014883
really    0.013818
time      0.013278
know      0.011402
even      0.011367
want      0.010689
much      0.010452
people    0.009284
bad       0.009141
Name: topic_8, dtype: float32

In [67]:
good_phi.iloc[:, 10].sort_values(ascending=False)[:20]  # TODO: wtf?

pt        0.026764
la        0.025257
period    0.018108
v         0.013092
pp        0.012353
bos       0.012262
det       0.010446
van       0.010058
cal       0.009803
chi       0.009781
pit       0.009757
nj        0.009474
tor       0.009454
min       0.009032
stl       0.008770
que       0.008429
play      0.008428
mon       0.008112
win       0.008061
power     0.008045
Name: topic_28, dtype: float32

In [75]:
good_phi.iloc[:, 15].sort_values(ascending=False)[:20]  # TODO: wtf?

db         0.075875
mov        0.017203
bh         0.015588
si         0.013994
al         0.013275
c          0.013219
byte       0.009442
bit        0.007018
di         0.006413
bl         0.006310
cx         0.005809
maxbyte    0.005197
mp         0.004885
mw         0.004685
mv         0.004409
inc        0.004321
pop        0.004132
mt         0.004039
mf         0.003946
loop       0.003771
Name: topic_41, dtype: float32

In [79]:
good_phi.iloc[:, 17].sort_values(ascending=False)[:20]

armenian      0.033250
turkish       0.018804
muslim        0.012646
turkey        0.010655
turk          0.009251
genocide      0.008986
greek         0.008893
people        0.008080
armenia       0.008001
government    0.006655
russian       0.006049
jew           0.005869
ottoman       0.005801
massacre      0.005548
soviet        0.005394
azerbaijan    0.005284
village       0.005281
army          0.005208
war           0.005092
history       0.005038
Name: topic_47, dtype: float32

## Ablation Study

In [64]:
DECORRELATION_TAU = BEST_TAUS[0]

ALL_PARAMS = [
    # (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0),

    (1, 0, 0),
    # (0, 1, 0),
    # (0, 0, 1),
]

In [65]:
SAVE_FOLDER + f'/ablation_study'

'results50/20newsgroups/ablation_study'

In [66]:
os.makedirs(SAVE_FOLDER + f'/ablation_study', exist_ok=True)

In [67]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

for params in ALL_PARAMS:
    key = params

    output_k = '-'.join(str(i) for i in key)
    res_file_path = SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue

    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    good_phi = None
    seed = 0

    os.makedirs(
        os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}'),
        exist_ok=True
    )

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        seed_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            print(f'Loading seed results from "{seed_save_folder}".')
            
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:
            # assert False
            
            custom_regularizers = dict()
            
            if params[0]:
                if good_phi is None:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_model=prev_model._model,
                        topic_names=good_topic_names,
                    )
                else:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_phi=good_phi,
                        topic_names=good_topic_names,
                    )
    
                print('test_2')
                
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            if params[1]:
                bad_phi = deepcopy(bad_phi)
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None

            if good_phi is None:
                good_phi = prev_model._model.get_phi()[good_topic_names]
    
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        output_k = '-'.join(str(i) for i in k)
        with open(SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

(1, 0, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.019165258246480757
sparse_theta_sp: -0.08944804715975409
decorrelation: 0.01
None
num_topics: {'good': 10, 'bad': 7, 'not_good': 40, 'total_bad': 7}
1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f04bb84ebe0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f04d4df8c70>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.023956572808100943
sparse_theta_sp: -0.1118100589496926
decorrelation: 0.01
fix: 1000000000
ext_decorr_good: 100000
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 15, 'bad': 10, 'not_good': 35, 'total_bad': 17}
Removing: results50/20newsgroups/ablation_study/iterative_100000_1-0-1/0
2
test_2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f04d4b71ca0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.03549121897496436
sparse_theta_sp: -0.16564453177732238
decorrelation: 0.01
fix: 1000000000
DOWNFALL: more bad topics...
num_topics: {'good': 23, 'bad': 11, 'not_good': 27, 'total_bad': 190}
Removing: results50/20newsgroups/ablation_study/iterative_100000_1-0-0/16
18
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f04c30f89d0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.03549121897496436
sparse_theta_sp: -0.16564453177732238
decorrelation: 0.01
fix: 1000000000
SUCCESS: less bad topics
num_topics: {'good': 23, 'bad': 9, 'not_good': 27, 'total_bad': 199}
Removing: results50/20newsgroups/ablation_study/iterative_100000_1-0-0/17
19
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f040e165ca0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.03549121897496436
sparse_theta_sp: -0.16564453177732238
decorrelation: 0.01
fix: 1000000000
DOWNFALL: more bad topics...
num_topics: {'good': 23, 'bad': 10, 'not_good': 27, 'total_bad': 209}
Removing: results50/20newsgroups/ablation_study/iterative_100000_1-0-0/18
Saving results


In [73]:
1

1

In [69]:
DECORRELATION_TAU = BEST_TAUS[1]

In [70]:
DECORRELATION_TAU

100000000

In [71]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

for params in ALL_PARAMS:
    key = params

    output_k = '-'.join(str(i) for i in key)
    res_file_path = SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue

    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    good_phi = None
    seed = 0

    os.makedirs(
        os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}'),
        exist_ok=True
    )

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        seed_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            print(f'Loading seed results from "{seed_save_folder}".')
            
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:
            # assert False
            
            custom_regularizers = dict()
            
            if params[0]:
                if good_phi is None:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_model=prev_model._model,
                        topic_names=good_topic_names,
                    )
                else:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_phi=good_phi,
                        topic_names=good_topic_names,
                    )
    
                print('test_2')
                
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            if params[1]:
                bad_phi = deepcopy(bad_phi)
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None

            if good_phi is None:
                good_phi = prev_model._model.get_phi()[good_topic_names]
    
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        output_k = '-'.join(str(i) for i in k)
        with open(SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

(1, 0, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.019165258246480757
sparse_theta_sp: -0.08944804715975409
decorrelation: 0.01
None
num_topics: {'good': 10, 'bad': 7, 'not_good': 40, 'total_bad': 7}
1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f04d5044a30>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f040d41bbb0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.023956572808100943
sparse_theta_sp: -0.1118100589496926
decorrelation: 0.01
fix: 1000000000
ext_decorr_good: 100000000
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 13, 'bad': 10, 'not_good': 37, 'total_bad': 17}
Removing: results50/20newsgroups/ablation_study/iterative2_100000000_1-0-1/0
2
test_2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f04c5a7ba90>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.03549121897496436
sparse_theta_sp: -0.16564453177732238
decorrelation: 0.01
fix: 1000000000
DOWNFALL: more bad topics...
num_topics: {'good': 23, 'bad': 11, 'not_good': 27, 'total_bad': 190}
Removing: results50/20newsgroups/ablation_study/iterative2_100000000_1-0-0/16
18
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f04bc7af0a0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.03549121897496436
sparse_theta_sp: -0.16564453177732238
decorrelation: 0.01
fix: 1000000000
SUCCESS: less bad topics
num_topics: {'good': 23, 'bad': 9, 'not_good': 27, 'total_bad': 199}
Removing: results50/20newsgroups/ablation_study/iterative2_100000000_1-0-0/17
19
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f04d53e33a0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.03549121897496436
sparse_theta_sp: -0.16564453177732238
decorrelation: 0.01
fix: 1000000000
DOWNFALL: more bad topics...
num_topics: {'good': 23, 'bad': 10, 'not_good': 27, 'total_bad': 209}
Removing: results50/20newsgroups/ablation_study/iterative2_100000000_1-0-0/18
Saving results


In [74]:
1

1

In [76]:
! ls results50/20newsgroups/ablation_study/ -alh

total 304K
drwxrwxr-x  8 alekseev_v mil_lab 4,0K мар 28 16:41 .
drwxrwxr-x  5 alekseev_v mil_lab 4,0K мар 28 14:48 ..
drwxrwxr-x 22 alekseev_v mil_lab 4,0K мар 28 15:36 iterative_100000_1-0-0
-rw-rw-r--  1 alekseev_v mil_lab  48K мар 28 15:36 iterative_100000_1-0-0.json
drwxrwxr-x 22 alekseev_v mil_lab 4,0K мар 28 15:06 iterative_100000_1-0-1
-rw-rw-r--  1 alekseev_v mil_lab  47K мар 28 15:36 iterative_100000_1-0-1.json
drwxrwxr-x 14 alekseev_v mil_lab 4,0K мар 28 15:17 iterative_100000_1-1-0
-rw-rw-r--  1 alekseev_v mil_lab  29K мар 28 15:36 iterative_100000_1-1-0.json
drwxrwxr-x 22 alekseev_v mil_lab 4,0K мар 28 16:41 iterative2_100000000_1-0-0
-rw-rw-r--  1 alekseev_v mil_lab  48K мар 28 16:41 iterative2_100000000_1-0-0.json
drwxrwxr-x 22 alekseev_v mil_lab 4,0K мар 28 15:57 iterative2_100000000_1-0-1
-rw-rw-r--  1 alekseev_v mil_lab  48K мар 28 16:41 iterative2_100000000_1-0-1.json
drwxrwxr-x 22 alekseev_v mil_lab 4,0K мар 28 16:19 iterative2_100000000_1-1-0
-rw-rw-r--  1 alekseev_

In [75]:
results.keys()

dict_keys([(0, 1, 1), (1, 0, 1), (1, 1, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1)])